# EDA

**Objectif**: analyser en profondeur le dataset pour orienter les décisions de chunking, prétraitement et retrieval dans le pipeline RAG;

**Dataset**: `dataset-tickets-multi-lang3-4k.csv` — 4 000 tickets, colonnes : `subject`, `body`, `answer`, `type`, `queue`, `priority`, `language`, `business_type`, `tag_1` … `tag_9`.

## 0. Imports et configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import re
import os

try:
    import plotly.express as px
    PLOTLY = True
except ImportError:
    PLOTLY = False
    print("[INFO] plotly non disponible — graphiques plotly ignorés")

try:
    from wordcloud import WordCloud
    WC = True
except ImportError:
    WC = False
    print("[INFO] wordcloud non disponible — WordCloud ignoré")

# Stopwords multilingues (EN/DE/FR/ES/PT)

STOPWORDS_EN = {
    'the','a','an','is','it','to','of','and','in','for',
    'i','my','we','you','your','our','this','that','with',
    'have','has','been','are','was','were','be','not','can',
    'will','would','could','should','please','hi','dear',
    'hello','regards','thank','thanks','mr','ms','mrs',
    'am','do','did','get','got','had','he','she','they',
    'me','us','him','her','its','or','but','if','as','at',
    'by','on','so','up','out','all','any','from','about',
    'into','than','more','also','just','need','use','used',
    'new','one','two','re','per','via','etc','may',
}

STOPWORDS_DE = {
    'ich','sie','er','es','wir','ihr','die','der','das',
    'ein','eine','ist','sind','war','haben','hat','nicht',
    'mit','für','auf','von','zu','an','in','bei','nach',
    'und','oder','aber','wenn','dass','wie','auch','noch',
    'sehr','bitte','mein','meine','sein','ihre',
    'habe','kann','wird','wurde','werden','anfrage',
    'im','am','den','dem','des','einen','einer',
    'sich','als','bis','aus','zum','zur','beim',
}

STOPWORDS_FR = {
    'je','tu','il','elle','nous','vous','ils','elles',
    'le','la','les','un','une','des','de','du','et','en',
    'est','sont','être','avoir','pas','plus','pour','sur',
    'avec','par','que','qui','quoi','dont','où','mais',
    'ou','si','car','donc','or','ni','ce','se','sa','son',
    'mon','ma','mes','votre','leur','bonjour','merci',
    'cordialement','madame','monsieur','cher','chère',
    'au','aux','y','ne','tout','bien','très','même',
}

STOPWORDS_ES = {
    'yo','él','ella','nosotros','vosotros','ellos',
    'el','la','los','las','un','una','unos','unas',
    'de','del','al','en','con','por','para','sobre',
    'es','son','está','estoy','tengo','tiene','hay',
    'que','qué','cómo','cuándo','dónde','quién',
    'pero','sin','si','ya','también','más','muy','no',
    'se','su','sus','mi','mis','tu','tus','le','les',
    'me','te','nos','estimado','estimada','saludos',
    'urgente','solicitud','problema','problemas',
}

STOPWORDS_PT = {
    'eu','tu','ele','ela','nós','vós','eles','elas',
    'o','a','os','as','um','uma','uns','umas',
    'de','do','da','dos','das','em','no','na','por',
    'para','com','sem','que','quê','como','quando',
    'é','são','está','tenho','tem','há','ser','ter',
    'mas','ou','se','já','também','mais','muito','não',
    'seu','sua','seus','suas','meu','minha',
    'prezado','prezada','atenciosamente','obrigado','obrigada',
}

# Termes ambigus, ou fragment sans valeur sémantique
STOPWORDS_EXTRA = {
    'dell',   # préposition italienne ET fragment "Dell XPS"
    'air',    # fragment "MacBook Air"
    'xps',    # modèle produit seul non informatif
    'pro',    # fragment "MacBook Pro"
    'http','https','www','com','nbsp',
}

# NLTK en complément si disponible
try:
    import nltk
    from nltk.corpus import stopwords as nltk_sw
    nltk.download('stopwords', quiet=True)
    STOPWORDS_EN |= set(nltk_sw.words('english'))
    STOPWORDS_DE |= set(nltk_sw.words('german'))
    STOPWORDS_FR |= set(nltk_sw.words('french'))
    STOPWORDS_ES |= set(nltk_sw.words('spanish'))
    STOPWORDS_PT |= set(nltk_sw.words('portuguese'))
    NLTK = True
    print("[INFO] nltk disponible — stopwords enrichis")
except ImportError:
    NLTK = False
    print("[INFO] nltk non disponible — stopwords manuels utilisés")

STOPWORDS_ALL = (
    STOPWORDS_EN | STOPWORDS_DE | STOPWORDS_FR |
    STOPWORDS_ES | STOPWORDS_PT | STOPWORDS_EXTRA
)

# Paramètres d'affichage
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print(f"Stopwords total : {len(STOPWORDS_ALL)} termes (EN/DE/FR/ES/PT)")
print("Imports OK")

In [ ]:
# Chargement du CSV

DATA_PATH = '../data/raw/dataset-tickets-multi-lang3-4k.csv'

df = pd.read_csv(DATA_PATH, encoding='utf-8')

TAG_COLS = [c for c in df.columns if c.startswith('tag_')]
TEXT_COLS = ['subject', 'body', 'answer']
CAT_COLS  = ['type', 'queue', 'priority', 'language', 'business_type']

print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(3)

In [ ]:
# Types des colonnes

df.dtypes.to_frame('dtype').T

## 1. Vue d'ensemble du dataset

Dimensions, types, valeurs nulles, unicité et doublons.

In [ ]:
# Dimensions et colonnes

print(f"Shape : {df.shape}")
print(f"Colonnes ({df.shape[1]}) : {list(df.columns)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Valeurs nulles

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_df = (
    pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
    .query('null_count > 0')
    .sort_values('null_pct', ascending=False)
)

print("Colonnes avec valeurs nulles :")
display(null_df)


# Heatmap pour un échantillon max 300 (illisible sinon)
sample_size = min(300, len(df))
sample_idx = np.linspace(0, len(df) - 1, sample_size, dtype=int)

null_matrix = df.isnull().iloc[sample_idx]

# Tri des colonnes par taux de null
cols_order = null_df.index
null_matrix = null_matrix[cols_order]

fig, ax = plt.subplots(figsize=(12, 4))

sns.heatmap(
    null_matrix.T,
    cmap='Greys',         
    cbar=False,
    yticklabels=True,
    xticklabels=False,
    ax=ax
)

# Titres
ax.set_title('Présence de valeurs manquantes par variable (échantillon)', fontsize=12, weight='bold')
ax.set_xlabel(f'Observations (n={sample_size})')
ax.set_ylabel('Variables')

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Valeurs uniques par colonne

unique_df = pd.DataFrame({
    'n_unique' : df.nunique(),
    'cardinality_pct': (df.nunique() / len(df) * 100).round(2)
}).sort_values('n_unique', ascending=False)

display(unique_df)

In [ ]:
# Doublons sur subject et body

n_dup = df.duplicated(subset=['subject', 'body']).sum()
print(f"Doublons exacts (subject + body) : {n_dup} ({n_dup/len(df)*100:.2f}%)")

if n_dup > 0:
    display(df[df.duplicated(subset=['subject', 'body'], keep=False)]
            .sort_values(['subject', 'body'])
            .head(6)[['subject', 'body', 'type', 'queue', 'priority', 'language']])

### Insights

- Les colonnes `tag_1` à `tag_9` sont naturellement sparse: les tags de rang élevé (tag_7, tag_8, tag_9) ont un taux de nullité important (attendu par conception).
- Les colonnes texte (`subject`, `body`, `answer`) et catégorielles (`type`, `queue`, `priority`, `language`) devraient être quasi-complètes; tout manquant constitue un défaut de qualité.
- L'absence de doublons (subject et body identiques) confirme la diversité des tickets.

## 2. Distribution des variables catégorielles

Exploration de `type`, `queue`, `priority`, `language`, `business_type` et leurs croisements.

In [ ]:
import matplotlib.pyplot as plt

# Distribution du type de tickets
type_counts = df['type'].value_counts().sort_values(ascending=False)

# Type de tickets majoritaire
max_type = type_counts.idxmax()

colors = ['#D3D3D3' if t != max_type else '#1976D2' for t in type_counts.index]

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(type_counts.index, type_counts.values, color=colors)

# Labels
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)

# Titre
ax.set_title('Répartition des tickets par type', fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Type de ticket')
ax.set_ylabel('Nombre de tickets')

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Rotation si nécessaire
plt.xticks(rotation=30, ha='right')

# Annotation du type dominant
ax.annotate(
    f"Majoritaire : {max_type}",
    xy=(0, type_counts.iloc[0]),
    xytext=(0, type_counts.iloc[0] * 1.1),
    textcoords='data',
    ha='center',
    fontsize=10,
    weight='bold',
    arrowprops=dict(arrowstyle='-')
)

plt.tight_layout()
plt.show()

# Tableau avec les pourcentages
summary = (
    type_counts
    .to_frame('count')
    .assign(pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1))
)

print(summary)

In [ ]:
import matplotlib.pyplot as plt

# Distribution queue (top 20)
queue_counts = df['queue'].value_counts().head(20).sort_values(ascending=True)

# Identification de la queue principale
max_queue = queue_counts.idxmax()

colors = ['#D3D3D3' if q != max_queue else '#1976D2' for q in queue_counts.index]

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(queue_counts.index, queue_counts.values, color=colors)

# Labels
labels = [
    f"{v} (max)" if q == max_queue else f"{v}"
    for q, v in zip(queue_counts.index, queue_counts.values)
]

ax.bar_label(bars, labels=labels, padding=3, fontsize=9)

# Titre
ax.set_title('Top 20 des files d’attente par volume de tickets', fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Nombre de tickets')
ax.set_ylabel('Queue')

# Grille
ax.xaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)



plt.tight_layout()
plt.show()

print(f"Nombre total de queues distinctes : {df['queue'].nunique()}")

In [ ]:
import matplotlib.pyplot as plt

# Distribution du niveau de priorité
priority_order = ['low', 'medium', 'high', 'critical']
prio_counts = (
    df['priority']
    .value_counts()
    .reindex([p for p in priority_order if p in df['priority'].unique()])
)

# Couleurs sémantiques
color_map = {
    'low': '#4CAF50',
    'medium': '#2196F3',
    'high': '#FF9800',
    'critical': '#F44336'
}
colors = [color_map[p] for p in prio_counts.index]

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(prio_counts.index, prio_counts.values, color=colors)

# Labels en pourcentage
total = prio_counts.sum()
labels = [f"{v} ({v/total*100:.1f}%)" for v in prio_counts.values]
ax.bar_label(bars, labels=labels, padding=3, fontsize=9)

# Titre
ax.set_title('Répartition des tickets par priorité', fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Priorité')
ax.set_ylabel('Nombre de tickets')

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Tableau
summary = (
    prio_counts
    .to_frame('count')
    .assign(pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1))
)

print(summary)

In [ ]:
import matplotlib.pyplot as plt

# Distribution de la langue des tickets
lang_counts = df['language'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 4))

# Une seule couleur
bars = ax.bar(lang_counts.index, lang_counts.values)

# Labels directement sur les barres
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)

# Titre
ax.set_title('Répartition des tickets par langue', fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Langue')
ax.set_ylabel('Nombre de tickets')

# Suppression des éléments inutiles
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

# Tableau avec les pourcentages
summary = (
    lang_counts
    .to_frame('count')
    .assign(pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1))
)

print(summary)

In [ ]:
import matplotlib.pyplot as plt

# Distribution du type de business (top 15)
bt_counts = (
    df['business_type']
    .value_counts()
    .head(15)
    .sort_values(ascending=True)
)

# Catégorie principale
max_bt = bt_counts.idxmax()
max_val = bt_counts.max()

colors = ['#D3D3D3' if b != max_bt else '#1976D2' for b in bt_counts.index]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(bt_counts.index, bt_counts.values, color=colors)

# Labels
labels = [
    f"{v} (max)" if b == max_bt else f"{v}"
    for b, v in zip(bt_counts.index, bt_counts.values)
]
ax.bar_label(bars, labels=labels, padding=3, fontsize=9)

# Titre
ax.set_title("Top 15 des types d’entreprise par volume de tickets", fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Nombre de tickets')
ax.set_ylabel('Business type')

# Grille
ax.xaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Ajustement
ax.set_xlim(0, max_val * 1.15)

plt.tight_layout()
plt.show()

print(f"Nombre total de business_type distincts : {df['business_type'].nunique()}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tabkeau croisé (pourcentage par type)
cross_tp = pd.crosstab(df['type'], df['priority'], normalize='index') * 100

# Colonnes ordonnées
ordered_prio = [p for p in priority_order if p in cross_tp.columns]
cross_tp = cross_tp[ordered_prio]

fig, ax = plt.subplots(figsize=(8, 4))

sns.heatmap(
    cross_tp,
    annot=True,
    fmt='.1f',
    cmap='Greys',          
    linewidths=0.5,
    cbar_kws={'label': '%'},
    ax=ax
)

# Titres
ax.set_title('Répartition des priorités par type (%)', fontsize=12, weight='bold')
ax.set_xlabel('Priorité')
ax.set_ylabel('Type')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Normalisation par ligne (chaque langue = 100%)
cross_lt = pd.crosstab(df['language'], df['type'], normalize='index')*100

fig, ax = plt.subplots(figsize=(8, 3))

sns.heatmap(
    cross_lt,
    annot=True,
    fmt='.1f',
    cmap='Greys',
    linewidths=0.5,
    cbar_kws={'label': '%'},
    ax=ax
)

ax.set_title('Répartition des types de tickets par langue (%)', fontsize=12, weight='bold')
ax.set_xlabel('Type')
ax.set_ylabel('Langue')

plt.tight_layout()
plt.show()

### Insights 

- Le croisement **type × priority** révèle que certains type sont sur-représentés (Incident, Request), ce qui peut biaiser un retrieval basé sur la fréquence.
- La langue est bien distribuée: le pipeline RAG doit supporter les **05 langues** avec un modèle d'embedding multilingue.
- Les queues (52 valeurs) offrent un filtre metadata pertinent pour le RAG: un utilisateur qui interroge sur la facturation pourrait filtrer `queue = Billing and Payments`.
- `business_type` à haute cardinalité peut servir de metadata secondaire pour personnaliser les réponses.

## 3. Analyse des tags

Fréquence, distribution, co-occurrence et répartition par type de ticket.

In [ ]:
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Collecte de tous les tags
all_tags = []

for col in TAG_COLS:
    all_tags.extend(
        df[col]
        .dropna()
        .astype(str)     
        .str.strip()
        .tolist()
    )

tag_freq = Counter(all_tags)

tag_df = (
    pd.DataFrame(tag_freq.most_common(30), columns=['tag', 'count'])
    .sort_values('count', ascending=True)
)

# Mise en évidence du top tag
max_tag = tag_df['tag'].iloc[-1]

colors = ['#D3D3D3' if t != max_tag else '#1976D2' for t in tag_df['tag']]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(tag_df['tag'], tag_df['count'], color=colors)

# Labels (avec pourcentage sur le top uniquement)
total_tags = len(all_tags)

labels = [
    f"{c} ({c/total_tags*100:.1f}%)" if t == max_tag else f"{c}"
    for t, c in zip(tag_df['tag'], tag_df['count'])
]

ax.bar_label(bars, labels=labels, padding=3, fontsize=9)

# Titre
ax.set_title('Top 30 des tags (toutes colonnes confondues)', fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Fréquence')
ax.set_ylabel('Tags')

# Grille
ax.xaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Stats
print(f"Nombre total de valeurs de tags : {len(all_tags):,}")
print(f"Valeurs de tags distinctes     : {len(tag_freq):,}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Nettoyage robuste: considère NaN + "" + "nan
df[TAG_COLS] = df[TAG_COLS].replace(["", "nan", "None"], pd.NA)

# Nombre de tags par ticket
df['n_tags'] = df[TAG_COLS].notna().sum(axis=1)

# Stats
print(f"Nombre moyen de tags par ticket : {df['n_tags'].mean():.2f}")
print(f"Médiane                         : {df['n_tags'].median():.0f}")

display(df['n_tags'].value_counts().sort_index().to_frame('count'))

# Distribution en pourcentage
dist = df['n_tags'].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(
    dist.index,
    dist.values,
    color='steelblue',
    edgecolor='white',
    width=0.8
)

# Labels en pourcentage
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)

# Titre
ax.set_title('Distribution du nombre de tags par ticket', fontsize=12, weight='bold')

# Axes
ax.set_xlabel('Nombre de tags')
ax.set_ylabel('Pourcentage de tickets')

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Axe
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

ticket_types = df['type'].dropna().unique()
top_n = 10

fig, axes = plt.subplots(1, len(ticket_types), figsize=(16, 5))

# Précaution en cas d'un seul type
if len(ticket_types) == 1:
    axes = [axes]

for ax, ttype in zip(axes, ticket_types):

    subset = df[df['type'] == ttype]

    tags_in_type = []

    for col in TAG_COLS:
        tags_in_type.extend(
            subset[col]
            .dropna()
            .astype(str) 
            .str.strip()
            .tolist()
        )

    top_tags = Counter(tags_in_type).most_common(top_n)

    if not top_tags:
        ax.set_title(f'Type: {ttype}\n(no data)')
        ax.axis('off')
        continue

    tags_names = [t[0] for t in top_tags]
    tags_counts = [t[1] for t in top_tags]

    # mise en évidence du top tag
    max_tag = tags_names[0]
    colors = ['#1976D2' if t == max_tag else '#D3D3D3' for t in tags_names]

    ax.barh(tags_names[::-1], tags_counts[::-1], color=colors[::-1])

    ax.set_title(f'Type: {ttype}', fontsize=11, weight='bold')
    ax.set_xlabel('Fréquence')

    # grille
    ax.xaxis.grid(True, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    # nettoyage
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(f'Top {top_n} tags par type de ticket', fontsize=13, y=1.05)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations

# 1. Nettoyage des tags

clean_tags = df[TAG_COLS].apply(
    lambda col: col.dropna().astype(str).str.strip()
)

# Liste de tags par ticket (set = pas de doublons par ticket)
df_tags = clean_tags.apply(lambda row: set(row.dropna()), axis=1)

# 2. Top 15 tags globaux

all_tags = []
for tags in df_tags:
    all_tags.extend(tags)

tag_freq = Counter(all_tags)
top15_tags = [t for t, _ in tag_freq.most_common(15)]

# 3. Co-occurrence efficace

pairs = Counter()

for tags in df_tags:
    filtered = [t for t in tags if t in top15_tags]
    pairs.update(combinations(sorted(filtered), 2))

## Matrice de co-occurrence

cooc = pd.DataFrame(0, index=top15_tags, columns=top15_tags)

for (a, b), v in pairs.items():
    cooc.loc[a, b] = v
    cooc.loc[b, a] = v

for i in range(len(top15_tags)):
    cooc.iloc[i, i] = 0


# 4. Normalisation de la matrice

cooc_norm = cooc / cooc.values.max()

# 5. Heatmap

fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(cooc_norm, dtype=bool))

sns.heatmap(
    cooc_norm,
    mask=mask,
    cmap='Greys',
    square=True,
    linewidths=0.3,
    cbar_kws={'label': 'Co-occurrence normalisée'},
    ax=ax
)

ax.set_title('Co-occurrence des 15 tags les plus fréquents', fontsize=12, weight='bold')

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

# 6. Stats utiles

print(f"Nombre total de tags (brut) : {len(all_tags):,}")
print(f"Tags uniques               : {len(tag_freq):,}")
print(f"Top 15 représentés         : {len(top15_tags)}")

### Insights sur les tags

- Les tags les plus fréquents correspondent aux thématiques dominantes du support IT.
- La co-occurrence révèle des regroupements thématiques naturels (ex. Outage + Performance, Security + Access) utiles pour définir des catégories de chunks.
- La distribution du nombre de tags par ticket indique si la plupart des tickets ont entre 5 et 6 tags, ce qui pourrait apporter une valeur sémantique supplémentaire.
- Les tags constituent d'excellentes **métadonnées de filtrage** pour le retrieval RAG (filtrer sur tag = 'Security' pour les requêtes de sécurité).

## 4. Analyse des longueurs de texte

Distribution des longueurs de `subject`, `body`, `answer` — clé pour calibrer la stratégie de chunking.

In [ ]:
# Calcul des longueurs
for col in TEXT_COLS:
    df[f'len_{col}'] = df[col].fillna('').str.len()
    df[f'tokens_{col}'] = (df[f'len_{col}'] / 4).astype(int)

LEN_COLS = [f'len_{c}' for c in TEXT_COLS]

# Statistiques descriptives
stats = df[LEN_COLS].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
print("Statistiques de longueur (caractères) :")
display(stats.round(0))

print("\nEstimation tokens (longueur / 4) :")
display(df[[f'tokens_{c}' for c in TEXT_COLS]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
).round(0))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ['#1976D2', '#388E3C', '#F57C00']

for ax, col, color in zip(axes, TEXT_COLS, colors):

    data = df[f'len_{col}'][df[f'len_{col}'] > 0].dropna()

    # Histogramme en densité
    ax.hist(
        data,
        bins='auto',           
        density=True,
        color=color,
        edgecolor='white',
        alpha=0.85
    )

    median = data.median()
    p95 = data.quantile(0.95)

    ax.axvline(median, color='black', linestyle='--', lw=1.5, label=f'Médiane={median:.0f}')
    ax.axvline(p95, color='darkred', linestyle=':', lw=1.5, label=f'P95={p95:.0f}')

    # Titre
    ax.set_title(f'Distribution des longueurs : {col}', fontsize=11, weight='bold')

    ax.set_xlabel('Nombre de caractères')
    ax.set_ylabel('Densité')

    # Grille
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    # Nettoyage
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.legend(fontsize=9)

fig.suptitle('Distribution des longueurs de texte (comparaison normalisée)', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

# Nettoyage des données
data_box = [
    df[f'len_{col}'][df[f'len_{col}'] > 0].dropna().values
    for col in TEXT_COLS
]

bp = ax.boxplot(
    data_box,
    labels=TEXT_COLS,
    patch_artist=True,
    showfliers=True,
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(linewidth=1),
    capprops=dict(linewidth=1)
)

# Couleurs
colors_bp = ['#D3D3D3', '#BDBDBD', '#9E9E9E']

for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)

# Titre 
ax.set_title('Distribution des longueurs de texte (caractères)', fontsize=12, weight='bold')

ax.set_ylabel('Nombre de caractères')
ax.set_xlabel('Champ texte')

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, cat in zip(axes, ['type', 'priority', 'language']):

    stats = (
        df.groupby(cat)['len_body']
        .agg(['mean', 'median'])
        .sort_values('mean', ascending=False)
    )

    x = stats.index
    width = 0.4
    pos = range(len(x))

    # Moyenne
    bars1 = ax.bar(
        [p - width/2 for p in pos],
        stats['mean'],
        width=width,
        label='Moyenne',
        color='#1976D2'
    )

    # Médiane
    bars2 = ax.bar(
        [p + width/2 for p in pos],
        stats['median'],
        width=width,
        label='Médiane',
        color='#90CAF9'
    )

    ax.set_title(f'Longueur du body par {cat}', fontsize=11, weight='bold')
    ax.set_xticks(pos)
    ax.set_xticklabels(x, rotation=20, ha='right')

    ax.set_ylabel('Caractères')

    ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 6))

# Echantillon
scatter_df = df[(df['len_body'] > 0) & (df['len_answer'] > 0)].sample(
    min(1500, len(df)),
    random_state=42
)

colors_type = {
    'Incident': '#E53935',
    'Request': '#1E88E5',
    'Problem': '#43A047',
    'Change': '#FB8C00'
}

for ttype, grp in scatter_df.groupby('type'):
    ax.scatter(
        grp['len_body'],
        grp['len_answer'],
        alpha=0.25,  
        s=12,
        color=colors_type.get(ttype, 'grey'),
        label=ttype
    )

# ligne de tendance globale
m, b = np.polyfit(scatter_df['len_body'], scatter_df['len_answer'], 1)
x = np.linspace(scatter_df['len_body'].min(), scatter_df['len_body'].max(), 100)
ax.plot(x, m*x + b, color='black', linewidth=2, label='Tendance globale')

# axes
ax.set_xlabel('Longueur body (caractères)')
ax.set_ylabel('Longueur answer (caractères)')
ax.set_title('Relation body vs answer (échantillon 1 500 tickets)', fontsize=12, weight='bold')

ax.legend(fontsize=9)

# Grille
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Tickets très courts et tickets très longs
very_short = df[df['len_body'] < 100]
very_long  = df[df['len_body'] > 2000]

print(f"Tickets body < 100 chars  : {len(very_short):,} ({len(very_short)/len(df)*100:.1f}%)")
print(f"Tickets body > 2000 chars : {len(very_long):,} ({len(very_long)/len(df)*100:.1f}%)")

print("\n--- Exemples de tickets très courts ---")
display(very_short[['subject', 'body', 'type', 'priority']].head(5))

print("\n--- Exemples de tickets très longs ---")
display(very_long[['subject', 'len_body', 'tokens_body', 'type', 'priority']].head(5))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CHUNK_SIZES = [256, 512, 1024]

print("Estimation de couverture selon la taille de chunk (tokens égalent à environ caractères / 4) :\n")

total = len(df)

for chunk_size in CHUNK_SIZES:
    fits = (df['tokens_body'] <= chunk_size).sum()
    print(
        f"  chunk_size={chunk_size:5d} → {fits:4d} tickets "
        f"({fits/total*100:.1f}%)"
    )


# Graphe

fig, ax = plt.subplots(figsize=(10, 4))

data = df['tokens_body'][df['tokens_body'] > 0]

ax.hist(
    data,
    bins=60,
    density=True,  
    color='steelblue',
    edgecolor='white',
    alpha=0.85
)

# lignes chunk
colors = ['green', 'orange', 'red']
for cs, lc in zip(CHUNK_SIZES, colors):
    ax.axvline(
        cs,
        color=lc,
        linestyle='--',
        linewidth=2,
        label=f'chunk={cs}'
    )

# Titres
ax.set_title('Distribution des longueurs de texte (tokens estimés)', fontsize=12, weight='bold')
ax.set_xlabel('Tokens (≈ caractères / 4)')
ax.set_ylabel('Densité')

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Insights sur la longueurs de texte

- La distribution des **tokens body** dicte la taille de chunk optimale: si la médiane est inférieure à 512 tokens, un chunk par ticket (body entier) peut suffire pour la majorité du corpus.
- Les tickets très courts (inférieure à 100 chars) constituent du bruit à filtrer ou à traiter séparément.
- Les tickets très longs (supérieurs à 2000 chars) nécessiteront un découpage; la stratégie recommandée, est un **sliding window avec overlap** de 10–15%.
- Le ratio longueur body/longueur answer indique si les réponses sont plus concises que les demandes (attendu) ou du même ordre de grandeur.
- La corrélation entre body et answer peut révéler si des tickets complexes génèrent des réponses plus détaillées.

## 5. Analyse linguistique

Répartition EN/DE/ES/FR/PT, longueurs par langue, exemples et structure des emails.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Top queues
top15_queues = df['queue'].value_counts().head(15).index

lang_queue = (
    df[df['queue'].isin(top15_queues)]
    .groupby(['queue', 'language'])
    .size()
    .unstack(fill_value=0)
)

# Pourcentages par queue
lang_queue_pct = lang_queue.div(lang_queue.sum(axis=1), axis=0) * 100

# Tri des queues
lang_queue_pct = lang_queue_pct.sort_values(by=lang_queue_pct.columns[0], ascending=True)

colors = cm.get_cmap('tab20', lang_queue_pct.shape[1])

fig, ax = plt.subplots(figsize=(12, 7))

lang_queue_pct.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    colormap='tab20',
    width=0.75
)

ax.set_title(
    f'Repartition des langues par queue (Top 15 queues)',
    fontsize=12,
    weight='bold'
)

ax.set_xlabel('Pourcentage (%)')
ax.set_ylabel('Queue')

# Grille
ax.xaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# légende
ax.legend(
    title='Langue',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    fontsize=9
)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Table en pourcentage
lang_type = df.groupby(['type', 'language']).size().unstack(fill_value=0)
lang_type_pct = lang_type.div(lang_type.sum(axis=1), axis=0) * 100

# Tri des types par volume total 
lang_type_pct = lang_type_pct.loc[
    lang_type.sum(axis=1).sort_values(ascending=True).index
]

n_langs = lang_type_pct.shape[1]
colors = cm.get_cmap('tab20', n_langs)

fig, ax = plt.subplots(figsize=(9, 5))

lang_type_pct.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    colormap='tab20',
    width=0.65
)

ax.set_title(
    f'Repartition des langues par type de ticket ({n_langs} langues)',
    fontsize=12,
    weight='bold'
)

ax.set_xlabel('Type de ticket')
ax.set_ylabel('Pourcentage (%)')

# Grille
ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Rotation
ax.tick_params(axis='x', rotation=15)

# légende
ax.legend(
    title='Langue',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    fontsize=9
)

# Nettoyage
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

languages = sorted(df['language'].dropna().unique())

# stats
len_lang = (
    df.groupby('language')['len_body']
    .agg(['mean', 'median', 'std', 'count'])
    .round(1)
)

print("Longueur body par langue :")
display(len_lang)

# layout
n = len(languages)
cols = min(n, 3)
rows = (n + cols - 1) // cols

colors = cm.get_cmap('tab10', n)

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]

for ax, lang, i in zip(axes, languages, range(n)):

    data = df[df['language'] == lang]['len_body']
    data = data[data > 0].dropna()

    # Histogramme en densité
    ax.hist(
        data,
        bins='auto',
        density=True,
        color=colors(i),
        edgecolor='white',
        alpha=0.85
    )

    median = data.median()

    ax.axvline(
        median,
        color='black',
        linestyle='--',
        linewidth=1.5,
        label=f'Médiane={median:.0f}'
    )

    ax.set_title(f'{lang.upper()} (n={len(data)})', fontsize=11, weight='bold')

    ax.set_xlabel('Caractères')
    ax.set_ylabel('Densité')

    # Grille
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    ax.legend(fontsize=9)

    # Nettoyage
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Suppression axes vides
for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

# 1. Exemples de tickets par langue

languages = sorted(df['language'].dropna().unique())

for lang in languages:
    sample = df[df['language'] == lang].head(2)

    print("=" * 60)
    print(f"EXEMPLES DE TICKETS — LANGUE : {lang.upper()}")
    print("=" * 60)

    for _, row in sample.iterrows():
        print(f"\n[Subject]  {row['subject']}")
        print(f"[Type]     {row['type']} | [Priority] {row['priority']} | [Queue] {row['queue']}")

        body_preview = str(row['body'])
        if len(body_preview) > 300:
            body_preview = body_preview[:300] + "..."

        print(f"[Body]     {body_preview}")
        print("-" * 60)

    print("\n")


# 2. Patterns (salutations + signatures)

salutation_patterns = [
    r'\b(dear|hello|hi|good morning|good afternoon|greetings)\b',
    r'\b(sehr geehrte|hallo|guten morgen|guten tag|liebe)\b',
    r'\b(bonjour|madame|monsieur|cher|chère|à qui de droit)\b',
    r'\b(hola|estimado|estimada|buenos días|buenas tardes)\b',
    r'\b(olá|prezado|prezada|bom dia|boa tarde|caro|cara)\b',
]

signature_patterns = [
    r'(best regards|kind regards|sincerely|thank you|thanks|regards)\s*$',
    r'(mit freundlichen grüßen|vielen dank|herzliche grüße|hochachtungsvoll)\s*$',
    r'(cordialement|bien cordialement|merci|salutations|veuillez agréer)\s*$',
    r'(atentamente|saludos cordiales|gracias|un saludo)\s*$',
    r'(atenciosamente|cumprimentos|obrigado|obrigada|com os melhores)\s*$',
]


# 3. Fonction de détection

def has_pattern(text, patterns):
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(re.search(p, text) for p in patterns)


# 4. Features

df['has_salutation'] = df['body'].apply(lambda x: has_pattern(x, salutation_patterns))
df['has_signature']  = df['body'].apply(lambda x: has_pattern(x, signature_patterns))


# 5. Stats globales

sal_rate = df['has_salutation'].mean() * 100
sig_rate = df['has_signature'].mean() * 100

print(f"\nTickets avec salutation détectée : {sal_rate:.1f}%")
print(f"Tickets avec signature détectée  : {sig_rate:.1f}%\n")


# 6. Analyse par langue

struct_by_lang = (
    df.groupby('language')[['has_salutation', 'has_signature']]
    .mean()
    .mul(100)
    .round(1)
    .rename(columns={
        'has_salutation': '% salutation',
        'has_signature': '% signature'
    })
    .sort_values('% signature', ascending=False)
)

display(struct_by_lang)


# 7. Visualisation

fig, ax = plt.subplots(figsize=(9, 4))

struct_by_lang.plot(
    kind='bar',
    ax=ax,
    color=['#42A5F5', '#EF5350'],
    width=0.65
)

ax.set_title('Présence de structure email par langue', fontsize=12, weight='bold')
ax.set_xlabel('Langue')
ax.set_ylabel('Pourcentage (%)')
ax.set_ylim(0, 100)

ax.yaxis.grid(True, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

ax.legend(title='Élément détecté')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Insights sur l'analyse des langues

- Le dataset contient **5 langues** (EN, DE, FR, ES, PT); un modèle d'embedding
  **monolingue ou bilingue EN/DE est insuffisant**.
- Modèle adapté: `paraphrase-multilingual-MiniLM-L12-v2` (384d, 50+ langues,
  léger) ou `paraphrase-multilingual-mpnet-base-v2` (768d, meilleure qualité).
- Si certaines queues sont mono-langue, un **filtre par langue** dans l'interface
  Streamlit améliorera la précision du retrieval.
- La présence de salutations et signatures (boilerplate) dans les emails implique
  un nettoyage en prétraitement; ces éléments dégradent la qualité sémantique
  des embeddings sans apporter de valeur informative.
- Les longueurs body varient selon la langue: calibrer le chunking sur le
  **P95 de la langue la plus verbeuse**.

## 6. Analyse de la qualité des données

Détection des anomalies, valeurs manquantes critiques, doublons et PII résiduels.

In [ ]:
# Tickets vides ou très courts
quality_issues = {}

quality_issues['body_empty']        = df['body'].isna().sum() + (df['body'].fillna('').str.strip() == '').sum()
quality_issues['body_very_short']   = (df['len_body'] < 50).sum()
quality_issues['answer_empty']      = df['answer'].isna().sum() + (df['answer'].fillna('').str.strip() == '').sum()
quality_issues['subject_empty']     = df['subject'].isna().sum() + (df['subject'].fillna('').str.strip() == '').sum()

for k, v in quality_issues.items():
    print(f"{k:30s} : {v:4d} tickets ({v/len(df)*100:.1f}%)")

In [ ]:
# Tickets potentiellement dupliqués (subject + queue)

dup_sq = df.duplicated(subset=['subject', 'queue'], keep=False).sum()
print(f"Tickets avec même subject ET même queue : {dup_sq} ({dup_sq/len(df)*100:.1f}%)")

if dup_sq > 0:
    dup_examples = (
        df[df.duplicated(subset=['subject', 'queue'], keep=False)]
        .sort_values(['subject', 'queue'])
        .head(6)[['subject', 'queue', 'type', 'priority', 'language']]
    )
    display(dup_examples)

In [ ]:
# Cohérence type × queue
# Combinaisons type × queue et leur fréquence
type_queue = df.groupby(['type', 'queue']).size().reset_index(name='count')

# Top combinaisons
print("Top 20 combinaisons type × queue :")
display(type_queue.sort_values('count', ascending=False).head(20))

# Combinaisons rares (potentiellement aberrantes)
rare = type_queue[type_queue['count'] <= 2]
print(f"\nCombinaisons type × queue avec ≤ 2 tickets : {len(rare)}")
if len(rare) > 0:
    display(rare.sort_values('count').head(10))

In [ ]:
# Détection de PII résiduels
PII_PATTERNS = {
    'email_address'    : r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}',
    'phone_number'     : r'(\+?\d[\d\s\-\.]{7,}\d)',
    'name_placeholder' : r'<name>',
    'placeholder_tag'  : r'\[\w+\]',
}

pii_results = {}
for label, pattern in PII_PATTERNS.items():
    matches_body   = df['body'].fillna('').str.contains(pattern, regex=True, case=False).sum()
    matches_answer = df['answer'].fillna('').str.contains(pattern, regex=True, case=False).sum()
    pii_results[label] = {'body': matches_body, 'answer': matches_answer}

pii_df = pd.DataFrame(pii_results).T
pii_df['total'] = pii_df['body'] + pii_df['answer']
pii_df['pct_body'] = (pii_df['body'] / len(df) * 100).round(1)

print("Détection de PII résiduels :")
display(pii_df)

# Exemples de tickets avec emails
mask_email = df['body'].fillna('').str.contains(
    PII_PATTERNS['email_address'], regex=True
)
if mask_email.sum() > 0:
    print(f"\nExemple body avec email détecté :")
    sample_row = df[mask_email].iloc[0]
    emails_found = re.findall(PII_PATTERNS['email_address'], str(sample_row['body']))
    print(f"  Emails trouvés : {emails_found[:3]}")

In [ ]:
# Résumé de l'analyse de la qualité des données

print("Résumé analyse de qualité")
print("=" * 50)
total_issues = sum(quality_issues.values())
print(f"Total tickets avec problèmes de qualité détectés : {total_issues}")
print(f"  dont body très court (<50c)  : {quality_issues['body_very_short']}")
print(f"  dont body vide               : {quality_issues['body_empty']}")
print(f"  dont answer vide             : {quality_issues['answer_empty']}")
print(f"  dont subject vide            : {quality_issues['subject_empty']}")
print(f"Dataset propre estimé          : {len(df) - total_issues} tickets")

### Insights Qualité des données

- Les **PII résiduels** (adresses email, numéros de téléphone, placeholders `<name>`) doivent être masqués ou supprimés avant l'indexation dans le vector store pour des raisons de confidentialité.
- Les tickets avec `body` vide ou <50 caractères sont à **exclure du corpus d'indexation**, car ils ne contribuent pas à la qualité du retrieval.
- Les combinaisons type × queue très rares (<3 occurrences) peuvent indiquer des erreurs de catégorisation ou des cas exceptionnels.
- Les `[PLACEHOLDER]` dans les corps d'emails indiquent des templates non instanciés; il faudra les filtrer en prétraitement.

## 7. Analyse du contenu texte

Vocabulaire dominant, mots-clés, structure type des tickets.

In [ ]:
# Tokenisation simple + stopwords

def tokenize(series, stopwords=None, max_tokens=None):
    """Retourne un Counter des mots après nettoyage."""
    text = ' '.join(series.fillna('').tolist()).lower()
    tokens = re.findall(r'\b[a-zA-ZäöüÄÖÜß]{3,}\b', text)
    if stopwords:
        tokens = [t for t in tokens if t not in stopwords]
    if max_tokens:
        tokens = tokens[:max_tokens]
    return Counter(tokens)

In [ ]:
# Top 30 mots subject

subject_counter = tokenize(df['subject'], stopwords=STOPWORDS_ALL)
top30_subject   = subject_counter.most_common(30)

words_s  = [w for w, _ in top30_subject]
counts_s = [c for _, c in top30_subject]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(words_s[::-1], counts_s[::-1], color=sns.color_palette('viridis', 30))
ax.set_title('Top 30 mots dans subject (hors stopwords)', fontsize=13)
ax.set_xlabel('Fréquence')
plt.tight_layout()
plt.show()

In [ ]:
# Top 30 mots body: on se limite à 500K tokens pour la performance

body_counter = tokenize(
    df['body'].sample(min(2000, len(df)), random_state=42),
    stopwords=STOPWORDS_ALL,
    max_tokens=500_000
)
top30_body   = body_counter.most_common(30)

words_b  = [w for w, _ in top30_body]
counts_b = [c for _, c in top30_body]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(words_b[::-1], counts_b[::-1], color=sns.color_palette('plasma', 30))
ax.set_title('Top 30 mots dans body (hors stopwords, échantillon 2 000)', fontsize=13)
ax.set_xlabel('Fréquence')
plt.tight_layout()
plt.show()

In [ ]:
# WordCloud subject

if WC:
    text_subject = ' '.join(
        [w for w, _ in subject_counter.most_common(200)]
    )
    wc = WordCloud(
        width=900, height=400,
        background_color='white',
        colormap='Blues',
        max_words=150
    ).generate_from_frequencies(dict(subject_counter.most_common(200)))

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('WordCloud - Sujets des tickets', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] wordcloud non disponible. Top 20 mots subject :")
    for w, c in subject_counter.most_common(20):
        print(f"  {w:25s}: {c}")

In [ ]:
# Phrases en début de body (10 premiers mots)

def first_n_words(text, n=10):
    if pd.isna(text):
        return ''
    words = str(text).strip().split()
    return ' '.join(words[:n]).lower()

first_words = df['body'].apply(first_n_words, n=10)
fw_counter  = Counter(first_words[first_words != ''])

print("Top 10 débuts de body (10 premiers mots) :")
for phrase, count in fw_counter.most_common(10):
    print(f"  ({count:4d}×) {phrase}")

In [ ]:
# Structure type d'un ticket: longueurs médianes

median_lens = {
    'subject (chars)' : df['len_subject'].median(),
    'body (chars)'    : df['len_body'].median(),
    'answer (chars)'  : df['len_answer'].median(),
    'subject (tokens)': df['tokens_subject'].median(),
    'body (tokens)'   : df['tokens_body'].median(),
    'answer (tokens)' : df['tokens_answer'].median(),
}

print("Structure médiane d'un ticket :")
for k, v in median_lens.items():
    print(f"  {k:25s}: {v:.0f}")

### Insights sur le contenu du texte

- Le vocabulaire dominant des sujets révèle les **thèmes clés** du corpus IT (password, access, network, invoice, error…), utiles pour construire des requêtes de test RAG représentatives.
- La répétition de formules d'ouverture en début de body (ex. "Dear support team, I am writing...") confirme la présence de boilerplate à nettoyer avant l'embedding.
- La structure médiane (5-10 tokens pour subjects, 100-300 tokens pour body, 100-250 tokens pour answer) guide le choix du **format de chunk**: inclure subject + body dans un chunk unique est raisonnable pour la majorité des tickets.

## 8. Insights pour le pipeline RAG

Synthèse des recommandations issues de l'analyse exploratoire.

### 8.1 Stratégie de chunking

| Paramètre | Recommandation | Justification (EDA) |
|---|---|---|
| **Unité de base** | 1 ticket = 1 chunk | Les tickets sont des unités sémantiques naturelles et autonomes (sujet + corps + réponse). Le découpage intra-ticket n'est justifié que pour les cas extrêmes. |
| **Contenu du chunk** | `Subject` + `\n\n` + `Body` + `\n\n` + `Answer` | La section 7 montre que subject (environ 5-10 tokens) enrichit le signal sémantique. `Answer` est inclus pour permettre le retrieval de type *« trouver une résolution similaire »*. |
| **Taille cible** | 512 tokens | La section 4 montre que la médiane `tokens_body` se situe en dessous de 512. Ce seuil couvre environ 80-90% du corpus sans troncature. |
| **Taille maximale** | 1 024 tokens | Le P95 des `tokens_body` est inférieur 1 024. Au-delà, le ticket est découpé en 2 chunks. Couvre plus de 95% du corpus. |
| **Overlap (si split)** | 50-75 tokens (environ 10-15%) | Evite la coupure d'une idée entre deux chunks. Valeur empirique standard pour la continuité contextuelle. |
| **Tickets trop courts** | Exclure si body inférieur à 50 chars | La section 6 identifie ces tickets comme du bruit (tickets vides, tests, erreurs de saisie); ils dégradent la qualité de l'index. |
| **Tickets sans réponse** | Exclure | Un ticket sans `answer` ne peut pas servir de précédent utile pour un agent SAV. |

> **Choix architectural** : on a opté pour la stratégie *1 ticket = 1 chunk* plutôt qu'un chunking fixed-size générique, car le dataset est composé d'entités documentaires naturellement délimitées. Un chunking par fenêtre glissante aurait introduit de la redondance et des chunks sémantiquement incohérents (ex. moitié d'une question + moitié d'une réponse).

In [ ]:
# Estimation du corpus final après filtrage qualité

df_clean_mask = (
    df['body'].notna() &
    (df['body'].str.strip() != '') &
    (df['len_body'] >= 50) &
    df['answer'].notna() &
    (df['answer'].str.strip() != '')
)

n_clean = df_clean_mask.sum()
print(f"Tickets éligibles après filtrage : {n_clean:,} / {len(df):,} ({n_clean/len(df)*100:.1f}%)")

tokens_clean = df.loc[df_clean_mask, 'tokens_body']
print(f"\nTokens body (corpus propre) :")
print(f"  Médiane : {tokens_clean.median():.0f} tokens")
print(f"  P75     : {tokens_clean.quantile(0.75):.0f} tokens")
print(f"  P95     : {tokens_clean.quantile(0.95):.0f} tokens")
print(f"  P99     : {tokens_clean.quantile(0.99):.0f} tokens")
print(f"  Max     : {tokens_clean.max():.0f} tokens")

n_split = (tokens_clean > 512).sum()
print(f"\nTickets nécessitant un split (>512 tokens) : {n_split} ({n_split/n_clean*100:.1f}% du corpus propre)")

# Répartition par langue dans le corpus propre
print(f"\nRépartition par langue (corpus propre) :")
print(df.loc[df_clean_mask, 'language'].value_counts().to_string())

In [ ]:
# Distribution tokens du corpus propre avec seuils de chunking

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(tokens_clean, bins=80, color='steelblue', edgecolor='white', alpha=0.85, label='Corpus propre')
ax.axvline(tokens_clean.median(), color='green', linestyle='--', lw=2,
           label=f'Médiane = {tokens_clean.median():.0f} tok')
ax.axvline(512,  color='orange', linestyle='-', lw=2, label='Seuil 512 tok (chunk cible)')
ax.axvline(1024, color='red',    linestyle='-', lw=2, label='Seuil 1024 tok (chunk max)')
ax.set_title('Distribution tokens estimés (body) — corpus propre après filtrage', fontsize=13)
ax.set_xlabel('Tokens estimés (longueur / 4)')
ax.set_ylabel('Fréquence')
ax.legend()
plt.tight_layout()
plt.show()

### 8.2 Métadonnées à indexer pour le filtrage

| Metadata | Cardinalité | Type OpenSearch | Usage RAG | Justification (EDA) |
|---|---|---|---|---|
| `queue` | ~52 valeurs | `keyword` | **Filtre primaire** | La section 2 montre une forte hétérogénéité des queues. Filtrer par queue évite de mélanger des domaines incompatibles (Billing ≠ Network). |
| `type` | 4 valeurs | `keyword` | Filtre secondaire | Incident ≠ Request sur le plan sémantique. Un agent cherchant un précédent d'Incident ne veut pas de Requests. |
| `priority` | 4 valeurs | `keyword` | Boost de score optionnel | Les tickets `critical` peuvent recevoir un poids supérieur dans le re-ranking si la requête est elle-même critique. |
| `language` | 5 valeurs | `keyword` | **Filtre obligatoire** | La section 5 confirme 5 langues (EN/DE/FR/ES/PT). Sans filtre langue, un agent FR recevrait des résultats DE/EN non pertinents, même avec un modèle multilingue. |
| `business_type` | Variable | `keyword` | Filtre optionnel | Personnalisation B2B : une entreprise de type *SaaS* a des tickets différents d'un *retailer*. |
| `tags` | ~30-50 valeurs | `keyword` (array) | Filtrage thématique fin | La section 3 montre que les tags couvrent des thèmes clés (Security, Outage, Billing). Permettre le filtrage multi-tags enrichit l'interface Streamlit. |

> **Choix architectural**: tous ces champs sont stockés comme `keyword` dans OpenSearch (pas analysés), ce qui permet un filtrage exact et rapide sans tokenisation. Le champ `text` (BM25) reste le seul champ analysé.

### 8.3 Langues et choix du modèle d'embedding

#### Constat (section 5)
Le dataset contient **5 langues** : EN, DE, FR, ES, PT. La répartition n'est pas uniforme; EN et DE dominent, mais FR/ES/PT sont présents en volume significatif.

#### Le modèle `all-MiniLM-L6-v2` est insuffisant car:
- Entraîné **principalement sur des corpus anglais** (Wikipedia EN, BookCorpus EN)
- Les embeddings DE/FR/ES/PT sont de qualité dégradée : la similarité cosinus entre tickets dans ces langues est moins fiable
- Risque : un ticket FR ne retrouvera pas un ticket FR similaire si les vecteurs sont mal positionnés dans l'espace sémantique

#### Modèle retenu pour le MVP : `paraphrase-multilingual-MiniLM-L12-v2`

| Critère | `all-MiniLM-L6-v2` | `paraphrase-multilingual-MiniLM-L12-v2` |
|---|---|---|
| Langues | EN principalement | **50+ langues** (EN, DE, FR, ES, PT inclus) |
| Dimension | 384 | **384** (compatible; pas besoin de recréer le mapping) |
| Taille modèle | 90 MB | 118 MB |
| MTEB score (multilingual) | Faible sur non-EN | Bon sur toutes les langues |
| Coût | Gratuit, local | Gratuit, local |
| Changement dans `.env` | — | `LOCAL_EMBEDDING_MODEL=paraphrase-multilingual-MiniLM-L12-v2` |

> **Avantage clé** : la dimension reste **384** — pas besoin de recréer le mapping OpenSearch ni de modifier `EMBEDDING_DIM` dans le `.env`.
Il suffit de changer `LOCAL_EMBEDDING_MODEL` et de relancer `pipeline_ingestion.py --recreate`.

#### Trajectoire production
- `paraphrase-multilingual-mpnet-base-v2` (768d) : meilleure qualité, ~278 MB
- `BAAI/bge-m3` (1024d) : état de l'art multilingue (100+ langues), dense + sparse, ~570 MB
- `text-embedding-3-large` via OpenRouter (3072d) : meilleure qualité absolue, payant ($0.13/1M tokens)

### 8.4 Prétraitement recommandé

| Étape | Action | Justification (EDA) |
|---|---|---|
| **Suppression boilerplate** | Retirer salutations et signatures (section 5) | Ces éléments répétitifs (*"Dear Support Team"*, *"Best regards"*) ne portent pas d'information sémantique utile et bruitent les embeddings. Ils sont détectés dans un fort pourcentage des tickets. |
| **Anonymisation PII** | Remplacer emails, téléphones, cartes par `[EMAIL]`, `[PHONE]`, `[CARD]` | La section 6 détecte des PII résiduels dans body/answer. Patterns étendus à EN/DE/FR/ES/PT (numéros de téléphone aux formats nationaux). |
| **Résolution placeholders** | `<name>` → supprimer ou remplacer par token neutre | Le dataset utilise déjà `<name>` comme anonymisation — le conserver tel quel évite de créer un faux signal. |
| **Normalisation espaces** | `re.sub(r'\\s+', ' ', text).strip()` | Les emails contiennent des sauts de ligne multiples, tabulations et espaces redondants issus du formatage HTML. |
| **Suppression HTML** | `re.sub(r'<[^>]+>', '', text)` | Certains tickets contiennent des balises HTML résiduelles (`<br>`, `<p>`, etc.). |
| **Conservation de la casse** | Ne pas mettre en minuscule | Les acronymes IT (VPN, DNS, API, SLA, LDAP…) sont porteurs de sens — les minusculiser les rend ambigus. Les modèles de sentence-transformers gèrent nativement la casse. |

#### Suppression du boilerplate — implémentation recommandée

```python
# Patterns de salutation à retirer (EN/DE/FR/ES/PT)
SALUTATION_RE = re.compile(
    r'^(dear|hello|hi|good morning|good afternoon|greetings|'
    r'sehr geehrte[rns]?|hallo|guten (morgen|tag)|liebe[rs]?|'
    r'bonjour|madame|monsieur|cher|chère|à qui de droit|'
    r'hola|estimad[oa]|buenos días|buenas tardes|'
    r'olá|prezad[oa]|bom dia|boa tarde)[,\\s\\w]*\\n+',
    re.IGNORECASE | re.MULTILINE
)

# Patterns de signature à retirer
SIGNATURE_RE = re.compile(
    r'(best regards|kind regards|sincerely|thank you|regards|'
    r'mit freundlichen grüßen|vielen dank|herzliche grüße|'
    r'cordialement|bien cordialement|merci|salutations|'
    r'atentamente|saludos cordiales|gracias|'
    r'atenciosamente|cumprimentos|obrigad[oa])[\\s\\S]*$',
    re.IGNORECASE
)

def remove_boilerplate(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = SALUTATION_RE.sub('', text)
    text = SIGNATURE_RE.sub('', text)
    return text.strip()
```

### 8.5 Justification de la recherche hybride BM25 + vectorielle

#### Pourquoi BM25 seul est insuffisant
- La section 7 montre que le vocabulaire IT est dense en acronymes et termes techniques (VPN, LDAP, DNS, API…)
- BM25 excelle pour retrouver un ticket contenant exactement *"VPN connection refused"* mais rate les tickets formulés *"unable to access remote network"* (même sens, termes différents)
- BM25 est insensible à la langue : il peut matcher des tokens FR et EN aléatoirement

#### Pourquoi vectoriel seul est insuffisant
- La recherche vectorielle ne retrouve pas les correspondances exactes sur des identifiants rares (numéros de ticket, codes produits, noms d'outils spécifiques) qui ont un vecteur dilué dans l'espace sémantique
- Coûteux en calcul sans GPU pour de grands corpus

#### Pourquoi RRF (Reciprocal Rank Fusion)
- Fusionne les listes BM25 et vectorielle sans nécessiter de calibration des scores bruts (BM25 et cosine sont dans des échelles incomparables)
- La formule `1 / (k + rank)` avec `k=60` est robuste et empiriquement validée ([Cormack et al., 2009](https://dl.acm.org/doi/10.1145/1571941.1572114))
- Chaque résultat expose son `bm25_rank` et `vector_rank` → **ranking explicable** pour l'agent SAV

#### Paramètres retenus
| Paramètre | Valeur | Justification |
|---|---|---|
| Candidats BM25 | 20 | Sur 4K tickets, 20 candidats couvrent largement les résultats pertinents |
| Candidats vectoriels | 20 | Idem |
| `k` RRF | 60 | Valeur standard de la littérature, évite de sur-pénaliser les documents absents d'une liste |
| Top-K final | 5 | Nombre de résultats affichés dans l'interface — compromis entre exhaustivité et lisibilité |

### 8.6 Justification du choix d'OpenSearch

OpenSearch a été retenu face à Qdrant pour les raisons suivantes :

| Critère | OpenSearch 2.13 | Qdrant 1.9 |
|---|---|---|
| **BM25 natif** | ✅ Intégré | ❌ Nécessite sparse vectors (SPLADE) |
| **Index vectoriel** | ✅ Plugin k-NN (HNSW) | ✅ Natif |
| **Un seul service** | ✅ BM25 + vectoriel dans le même index | ❌ Nécessite un composant BM25 externe |
| **Complexité infra MVP** | Faible (1 container) | Moyenne (2 containers) |
| **Licence** | Apache 2.0 | Apache 2.0 |
| **Dashboard** | ✅ OpenSearch Dashboards inclus | ❌ Web UI limitée |

> **Décision** : pour un MVP en 10 jours avec un seul container à gérer, OpenSearch est le choix optimal. Qdrant sera envisagé en trajectoire production si les besoins en filtrage vectoriel avancé ou en scalabilité horizontale l'exigent.

### 8.7 Valeurs aberrantes et actions correctives

| Anomalie | Détection (section EDA) | Action dans le pipeline |
|---|---|---|
| `body` < 50 chars | Section 6 — `quality_issues['body_short']` | Exclusion avant chunking (`preprocessor.py`) |
| `body` > 2 000 chars (>P95) | Section 4 — `very_long` | Split en 2 chunks avec overlap 75 tokens (`chunker.py`) |
| `answer` vide | Section 6 — `quality_issues['answer_empty']` | Exclusion — ticket sans résolution non indexable |
| Doublons `subject+body` | Section 1 — `n_dup` | `drop_duplicates(subset=['subject','body'])` dans preprocessor |
| Doublons `subject+queue` | Section 6 — `dup_sq` | Inspection manuelle — peut être légitime (même sujet, queues différentes) |
| PII résiduels (email, tel) | Section 6 — `pii_counts` | `anonymize_pii()` dans preprocessor (patterns EN/DE/FR/ES/PT) |
| Placeholder `<name>` | Section 6 | Conserver tel quel (déjà anonymisé) |
| Combinaisons type×queue rares | Section 6 | Logguer pour investigation, ne pas exclure automatiquement |

In [ ]:
# Tableau récapitulatif des recommandations
import pandas as pd

reco = pd.DataFrame([
    ('Modèle embedding',    'paraphrase-multilingual-MiniLM-L12-v2', 'Gratuit, local, 50+ langues, dim=384'),
    ('Dimension vecteur',   '384',                                   'Compatible avec index OpenSearch existant'),
    ('Chunk size cible',    '512 tokens',                            f'Couvre ~{(tokens_clean <= 512).mean()*100:.0f}% du corpus propre'),
    ('Chunk size max',      '1 024 tokens',                          f'Couvre ~{(tokens_clean <= 1024).mean()*100:.0f}% du corpus propre'),
    ('Overlap (si split)',  '75 tokens',                             'Continuité contextuelle entre chunks'),
    ('Contenu chunk',       'subject + body + answer',               'Unité sémantique complète par ticket'),
    ('Filtre qualité min',  'body >= 50 chars + answer non vide',    f'{n_clean:,} tickets éligibles ({n_clean/len(df)*100:.1f}%)'),
    ('Moteur indexation',   'OpenSearch 2.13',                       'BM25 + k-NN dans un seul service'),
    ('Fusion',              'RRF (k=60)',                            'Fusion sans calibration, ranking explicable'),
    ('Top-K interface',     '5 résultats',                           'Compromis exhaustivité / lisibilité'),
    ('Filtre langue',       'Obligatoire (5 langues EN/DE/FR/ES/PT)','Évite les cross-langue non pertinents'),
    ('Filtre queue',        'Primaire (52 valeurs)',                  'Évite le mélange de domaines incompatibles'),
], columns=['Paramètre', 'Valeur retenue', 'Justification'])

reco.index = reco.index + 1
display(reco.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
))

In [ ]:
# Visualisation récapitulative: distribution tokens + seuils

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gauche: distribution tokens corpus propre
ax = axes[0]
ax.hist(tokens_clean, bins=80, color='steelblue', edgecolor='white', alpha=0.85, label='Corpus propre')
ax.axvline(tokens_clean.median(), color='green',  linestyle='--', lw=2, label=f'Médiane={tokens_clean.median():.0f}')
ax.axvline(512,  color='orange', linestyle='-', lw=2, label='512 tok (cible)')
ax.axvline(1024, color='red',    linestyle='-', lw=2, label='1024 tok (max)')
ax.set_title('Tokens estimés (body) — corpus propre', fontsize=12)
ax.set_xlabel('Tokens estimés')
ax.set_ylabel('Fréquence')
ax.legend(fontsize=9)

# Droite: répartition par langue dans le corpus propre
ax2 = axes[1]
lang_clean = df.loc[df_clean_mask, 'language'].value_counts()
colors_lang = plt.cm.tab10.colors[:len(lang_clean)]
bars = ax2.bar(lang_clean.index.str.upper(), lang_clean.values, color=colors_lang, edgecolor='white', width=0.6)
for bar, val in zip(bars, lang_clean.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:,}\n({val/n_clean*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax2.set_title('Répartition par langue (corpus propre)', fontsize=12)
ax2.set_xlabel('Langue')
ax2.set_ylabel('Nombre de tickets')
ax2.set_ylim(0, lang_clean.max() * 1.2)

plt.suptitle('Récapitulatif EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nEDA terminée.')
print('   - Mettre à jour LOCAL_EMBEDDING_MODEL dans .env')
print('   - Relancer: python -m scripts.pipeline_ingestion --recreate')